<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/HW/OwenAbbata/HW2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

In [20]:
def solve_bridge_circuit(V0, R1, R2, R3, R4, R5):
  """Solve a bridge circuit with five resistors and one voltage source.

    The circuit has the topology:
        B ---- R4 ---- C--(-)
        | L            |
        |  L           |
        R3  L__R5 _    R2
        |           L  |
        |            L |
    (+)-A ---- R1 ---- D


    Uses Kirchhoff's current and voltage laws to set up
    a system of linear equations A @ I = b.

    Parameters
    ----------
    V0 : float
        Voltage of the source (V).
    R1 : float
        Resistance of first resistor (Ohm).
    R2 : float
        Resistance of second resistor (Ohm).
    R3 : float
        Resistance of third resistor (Ohm).
    R4 : float
        Resistance of second resistor (Ohm).
    R5 : float
        Resistance of third resistor (Ohm).

    Returns
    -------
    currents : numpy.ndarray
        Array [I1, I2, I3, I4, I5] of branch currents (A).
    """
  """ Assume current directions:
   I1: V0 -> R1 -> Node B
   I2: Node B -> R2 -> Node C (Ground)
   I3: V0 -> R3 -> Node D
   I4: Node D -> R4 -> Node C (Ground)
   I5: Node B -> R5 -> Node D
  """
  """
   Formulate equations based on KCL at nodes B and D, and KVL for three independent loops.
   Equations:
   1. KCL at Node B: I1 - I2 - I5 = 0
   2. KCL at Node D: I3 + I5 - I4 = 0
   3. KVL loop V0-R1-R2-C-V0: V0 - I1*R1 - I2*R2 = 0  => R1*I1 + R2*I2 = V0
   4. KVL loop V0-R3-R4-C-V0: V0 - I3*R3 - I4*R4 = 0  => R3*I3 + R4*I4 = V0
   5. KVL loop B-R5-D-R4-R2-B: I5*R5 + I4*R4 - I2*R2 = 0 => -R2*I2 + R4*I4 + R5*I5 = 0
  """

  A = np.array([
      [1, -1, 0, 0, -1],  # KCL at Node B (I1 - I2 - I5 = 0)
      [0, 0, 1, -1, 1],   # KCL at Node D (I3 - I4 + I5 = 0)
      [R1, R2, 0, 0, 0],  # KVL Loop A-B-C-A (R1*I1 + R2*I2 = V0)
      [0, 0, R3, R4, 0],  # KVL Loop A-D-C-A (R3*I3 + R4*I4 = V0)
      [0, -R2, 0, R4, R5] # KVL Loop B-C-D-B (-R2*I2 + R4*I4 + R5*I5 = 0)
  ], dtype=float)

  b = np.array([0, 0, V0, V0, 0], dtype=float)

  print("Matrix A:")
  print(A)
  print("Matrix b:")
  print(b)

  currents = np.linalg.solve(A, b)
  return currents


# Given Values
V0 = 10  # Volts
R1 = 100  # Ohm
R2 = 200  # Ohm
R3 = 300  # Ohm
R4 = 600  # Ohm
R5 = 50   # Ohm

currents = solve_bridge_circuit (V0, R1, R2, R3, R4, R5)

print("Branch currents:")
print(f"I1 = {currents[0]:.4f} A  (through R1)")
print(f"I2 = {currents[1]:.4f} A  (through R2)")
print(f"I3 = {currents[2]:.4f} A  (through R3)")
print(f"I4 = {currents[3]:.4f} A  (through R4)")
print(f"I5 = {currents[4]:.4f} A  (through R5)")

Matrix A:
[[   1.   -1.    0.    0.   -1.]
 [   0.    0.    1.   -1.    1.]
 [ 100.  200.    0.    0.    0.]
 [   0.    0.  300.  600.    0.]
 [   0. -200.    0.  600.   50.]]
Matrix b:
[ 0.  0. 10. 10.  0.]
Branch currents:
I1 = 0.0333 A  (through R1)
I2 = 0.0333 A  (through R2)
I3 = 0.0111 A  (through R3)
I4 = 0.0111 A  (through R4)
I5 = 0.0000 A  (through R5)


In [21]:
def verify_power_balance(V0, R1, R2, R3, R4, R5, currents):
  """Verify conservation of energy in the circuit.

    #Checks that the power delivered by the voltage source equals the total power dissipated in all resistors.

    Parameters
    ----------
    V0 : float
        Voltage of the source (V).
    R1, R2, R3 : float
        Resistances (Ohm).
    currents : numpy.ndarray
        Array [I1, I2, I3] of branch currents (A).

    Returns
    -------
    bool variable
        True if power balance holds (within numerical tolerance).
    """
  I1, I2, I3, I4, I5 = currents
  P_source = V0 * (I1 + I3)
  P_dissipated = I1**2 * R1 + I2**2 * R2 + I3**2 * R3 + I4**2 * R4 + I5**2 * R5

  print(f"Power delivered by source:     {P_source:.4f} W")
  print(f"Power dissipated in resistors: {P_dissipated:.4f} W")

  return np.isclose(P_source, P_dissipated)


balanced = verify_power_balance(V0, R1, R2, R3, R4, R5, currents)
print(f"Energy conserved: {balanced}")

Power delivered by source:     0.4444 W
Power dissipated in resistors: 0.4444 W
Energy conserved: True


In [22]:
def verify_bridge_balance( R1, R2, R3, R4 ):
  """Verify circuit bridge is balanced.

    #Checks that no current is flowing in R5 by seeing is R1/R3 = R2/R4 if R3 = 0 or R4 = 0 it will be set to false

    Parameters
    ----------
    R1, R2, R3, R4: float
        Resistances (Ohm).


    Returns
    -------
    bool variable
        True if circuit is balanced
    """
  if R3 == 0 or R4 == 0:
    print("Cannot determine balance if R3 or R4 is zero.")
    return False

  is_balanced = np.isclose(R1 / R3, R2 / R4)
  print(f"Bridge is balanced: {is_balanced}")
  return is_balanced

balanced_check = verify_bridge_balance(R1, R2, R3, R4)


Bridge is balanced: True
